In [ ]:
///- --package ..
:< sm'_operators
:< engine

#### page (test)

In [ ]:
///- --test static

:< testing

In [1]:
:| kind =
    | Article
    | Release

:== author = { name : string; handle : string }

:== entry =
    {
        slug : string
        title : string
        date : string
        kind : kind
        authors : list author
        blurb : string
        blocks : list post.block
    }

:= kind_label (k : kind) : string =
    ?= k ?|
    | Article => ""
    | Release => "Release"

:= ^ join_names (authors : list author) : string =
    ?= authors ?|
    | [] => ""
    | [ a ] => html.escape a.name
    | a :: rest => html.escape a.name ++# ", " ++# join_names rest

:= ^ render_blocks (section : i32) (blocks : list post.block) : string =
    ?= blocks ?|
    | [] => ""
    | post.Heading text :: rest =>
        "<h2 id=\"section-" ++#? section ++# "\">" ++# html.escape text ++# "</h2>" ++# render_blocks (section + 1) rest
    | b :: rest => post.render_block b ++# render_blocks section rest

:= ^ toc_items (section : i32) (blocks : list post.block) : string =
    ?= blocks ?|
    | [] => ""
    | post.Heading text :: rest =>
        "<li><a href=\"#section-"
        ++#? section
        ++# "\">"
        ++# html.escape text
        ++# "</a></li>"
        ++# toc_items (section + 1) rest
    | _ :: rest => toc_items section rest

:= table_of_contents (blocks : list post.block) : string =
    := items = toc_items 1 blocks
    ? items = "" ?> "" |: "<nav class=\"toc\"><p>On this page</p><ol>" ++# items ++# "</ol></nav>"

:= render_entry (e : entry) : string =
    "<article><h1 class=\"post-title\">" ++# html.escape e.title ++# "</h1>"
    ++# "<p class=\"byline\">" ++# join_names e.authors ++# " &middot; " ++# html.escape e.date ++# "</p>"
    ++# table_of_contents e.blocks ++# render_blocks 1 e.blocks ++# "</article>"

:= sample () : entry =
    {
        slug = "sample"
        title = "Sample Entry"
        date = "2026-10-17"
        kind = Article
        authors = [ { name = "i574n"; handle = "i574n" } ]
        blurb = "A fixture for the engine's own tests."
        blocks = [ post.Heading "The Matrix"; post.Rule ]
    }

#### an_article_has_no_type_label (test)

In [2]:
///- --test

kind_label Article |> _assert_eq ""

__assert_eq { expected =  }

#### the_byline_lists_authors_and_date (test)

In [3]:
///- --test

render_entry (sample ()) |> sm'.contains "<p class=\"byline\">i574n &middot; 2026-10-17</p>" |> _assert_eq !+

__assert_eq { expected = true }

#### blocks_render_in_order (test)

In [4]:
///- --test

render_entry (sample ())
|> sm'.contains "<h2 id=\"section-1\">The Matrix</h2><hr class=\"section-rule\">"
|> _assert_eq !+

__assert_eq { expected = true }

#### the_table_of_contents_links_every_heading (test)

In [5]:
///- --test

table_of_contents [ post.Heading "A"; post.Paragraph "x"; post.Heading "B" ]
|> _assert_eq (
    "<nav class=\"toc\"><p>On this page</p><ol><li><a href=\"#section-1\">A</a></li><li><a "
    ^^ "href=\"#section-2\">B</a></li></ol></nav>"
)

__assert_eq { expected = <nav class="toc"><p>On this page</p><ol><li><a href="#section-1">A</a></li><li><a href="#section-2">B</a></li></ol></nav> }

#### a_post_without_headings_has_no_table_of_contents (test)

In [6]:
///- --test

table_of_contents [ post.Paragraph "x" ] |> _assert_eq ""

__assert_eq { expected =  }